# 047 — Métricas, calibración, sesgo y costo de error

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Matriz de confusión:** precision = TP/(TP+FP), recall (TPR) = TP/(TP+FN),
FPR = FP/(FP+TN), F1 = media armónica. La accuracy engaña con clases desbalanceadas:
leer siempre contra prevalencia y baseline.

**Curvas:** ROC (TPR vs FPR; AUC = P(score⁺ > score⁻), insensible a prevalencia) vs.
**PR** (la honesta con clase rara: su baseline es la prevalencia).

**Calibración:** p̂ = 0.7 debe implicar ~70 % de positivos.
`Brier = (1/n)Σ(p̂−y)² = incertidumbre − resolución + descalibración`. AUC mide solo
ranking; Brier/log-loss miden ranking y calibración.

**Costos:** con p̂ calibrada, umbral óptimo `t* = C_FP/(C_FP+C_FN)`; comparar modelos por
costo total, no por accuracy.

**Equidad:** medir TPR/FPR/calibración POR subgrupo. Teorema de imposibilidad: con
prevalencias distintas, calibración por grupo e igualdad de errores no pueden coexistir —
elegir y documentar es una decisión normativa.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** n = 1000. Accuracy = 970/1000 = **0.97**; precision = 30/50 = **0.60**;
recall = 30/40 = **0.75**; FPR = 20/960 ≈ **0.021**; F1 = 2·0.6·0.75/1.35 ≈ **0.667**.
El baseline "todo negativo" logra accuracy 0.96 — casi igual — con recall 0: la accuracy
apenas distingue el modelo del trivial; precision/recall sí lo hacen.

**Ejercicio 2.** Brier A = [(0.1)² + (0.2)² + (0.3)² + (0.2)² + (0.1)²]/5 =
[0.01+0.04+0.09+0.04+0.01]/5 = **0.038**. Brier B = [(0.4)²·5]/5 = **0.16**. El Brier
premia la *confianza correcta*: B acierta siempre pero con probabilidades tibias (0.6/0.4),
casi inservibles para decidir con costos; A separa de verdad.

**Ejercicio 3.** (a) t* = 50/500 = **0.1**. (b) Costo(0.5) = 12·50 + 30·450 =
600 + 13 500 = **14 100**; costo(t*) = 48·50 + 8·450 = 2400 + 3600 = **6 000**: gana t*,
menos de la mitad del costo, aceptando 4× más falsos positivos para evitar 22 falsos
negativos caros. (c) Que p̂ esté **calibrada**: t* = C_FP/(C_FP+C_FN) es óptimo solo si
p̂ es la probabilidad real; si no, el umbral óptimo empírico se busca barriendo sobre
validación (como en (b)).

**Ejercicio 4.** Grupo A: recall = 3/3 = **1.00**, FPR = 0/1 = **0.00**. Grupo B:
recall = 0/1 = **0.00**, FPR = 1/3 ≈ **0.33**. El agregado 0.75/0.75 promedia un grupo
donde el modelo es perfecto con otro donde es peor que inútil: viola la igualdad de
oportunidades (TPR igual por grupo) y la igualdad de odds. Con n tan pequeño además nada
es concluyente — exactamente las dos limitaciones que el laboratorio declara.


In [ ]:
result = run_lab("evaluation", seed=47)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — métricas y Brier
def metricas(tp, fp, fn, tn):
    n = tp + fp + fn + tn
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    return {
        "accuracy": (tp + tn) / n,
        "precision": p,
        "recall": r,
        "fpr": fp / (fp + tn),
        "f1": 2 * p * r / (p + r),
    }

for k, v in metricas(30, 20, 10, 940).items():
    print(f"{k:>9}: {v:.3f}")

def brier(y, p):
    return sum((pi - yi) ** 2 for yi, pi in zip(y, p)) / len(y)

y   = [1, 0, 1, 1, 0]
p_a = [0.9, 0.2, 0.7, 0.8, 0.1]
p_b = [0.6, 0.4, 0.6, 0.6, 0.4]
print(f"Brier A = {brier(y, p_a):.3f}   Brier B = {brier(y, p_b):.3f}")


In [ ]:
# Ejercicios 3 y 4 — costos y disparidad por grupo
c_fp, c_fn = 50, 450
print(f"t* = {c_fp / (c_fp + c_fn):.2f}")
print(f"costo umbral 0.5: {12 * c_fp + 30 * c_fn}")
print(f"costo umbral t* : {48 * c_fp + 8 * c_fn}")

result = run_lab("evaluation", seed=47)
print("agregado:", result["result"])
grupos = {"A": dict(tp=3, fp=0, fn=0, tn=1), "B": dict(tp=0, fp=1, fn=1, tn=2)}
for g, c in grupos.items():
    recall = c["tp"] / (c["tp"] + c["fn"]) if c["tp"] + c["fn"] else float("nan")
    fpr = c["fp"] / (c["fp"] + c["tn"]) if c["fp"] + c["tn"] else float("nan")
    print(f"grupo {g}: recall={recall:.2f}  FPR={fpr:.2f}")
# El 0.75/0.75 agregado esconde recall 1.00 vs 0.00: desagregar es obligatorio.


## Reflexión

1. El laboratorio reporta precision = recall = 0.75 sobre 8 ejemplos y lo declara en sus
   limitaciones. Calcula un intervalo aproximado para ese 0.75 (piensa en ±1/√n) y explica
   qué afirmaciones quedan fuera del alcance de la evidencia.
2. Dos modelos tienen el mismo AUC-ROC = 0.85, pero uno produce p̂ concentradas en
   [0.4, 0.6] y el otro en [0.05, 0.95]. ¿Cuál preferirías para decidir con matriz de
   costos y qué verificación harías antes?
3. El laboratorio menciona "falta análisis por grupos y costo" en `limitations`. Diseña el
   experimento mínimo que las cubriría: qué desagregarías y qué dos números nuevos
   reportarías.
